# 🏦 05. End-to-End Banking Fraud Operations Simulator
### Project: XAI System in Banking: AI Anomaly-Driven Credit Card Fraud Detection and Interpretation
**Objective:** Interactive demonstration simulating a real-time banking transaction stream. The system evaluates incoming transactions through the dual-stage Anomaly + Deep Learning pipeline, determines actionable banking protocol tiers, and automatically formats Bank Security Operations Alert Cards.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, torch

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.data.preprocessing import DataPipeline
from src.models.autoencoder import DeepAutoencoder
from src.xai.analyst_report import BankingAnalystReporter
from src.utils.device import get_device

device = get_device("auto")
print(f"Device: {device}")

## 1. Load Production Artifacts

In [ ]:
pipeline = DataPipeline(processed_dir=str(ROOT_DIR / "data/processed"))
data = pipeline.load_processed()

xgb_model = joblib.load(ROOT_DIR / "results/models/xgboost.joblib")
ae_ckpt = torch.load(ROOT_DIR / "results/models/autoencoder.pt", map_location=device, weights_only=False)

autoencoder = DeepAutoencoder(input_dim=data.X_test.shape[1], latent_dim=8).to(device)
autoencoder.load_state_dict(ae_ckpt["model_state_dict"] if "model_state_dict" in ae_ckpt else ae_ckpt)
threshold = ae_ckpt.get("threshold", 0.015)
reporter = BankingAnalystReporter(output_dir=str(ROOT_DIR / "results/explanations"))
print("All production models successfully initialized.")

## 2. Simulate Transaction Processing Stream
We simulate incoming transaction evaluation:
1. Fast Autoencoder Reconstruction Check
2. Deep Classifier Probability Scoring
3. Dynamic Protocol Recommendation
4. Instant Analyst Card Compilation

In [ ]:
def process_transaction(txn_id: str, x_vector: np.ndarray):
    # 1. Anomaly Check
    with torch.no_grad():
        t_x = torch.as_tensor(x_vector.reshape(1, -1), dtype=torch.float32, device=device)
        recon_err = float(autoencoder.compute_reconstruction_error(t_x, reduction="mean").cpu().numpy()[0])
    
    # 2. Classifier Probability
    prob = float(xgb_model.predict_proba(x_vector.reshape(1, -1))[0, 1])
    
    # 3. Simple SHAP attribution placeholder
    shap_vals = (x_vector * -0.2)
    
    card = reporter.generate_card(
        transaction_id=txn_id,
        probability=prob,
        feature_names=data.feature_names,
        feature_values=x_vector,
        shap_values=shap_vals,
        recon_error=recon_err,
        threshold=threshold,
    )
    
    md = reporter.render_markdown(card, filename=f"demo_card_{txn_id}.md")
    return card, md

## 3. Case Study A: Detected Fraud Transaction

In [ ]:
fraud_idx = np.where(data.y_test == 1)[0][0]
card_a, md_a = process_transaction("CASE-001-FRAUD", data.X_test[fraud_idx])
from IPython.display import Markdown
Markdown(md_a)

## 4. Case Study B: Legitimate Everyday Purchase

In [ ]:
normal_idx = np.where(data.y_test == 0)[0][0]
card_b, md_b = process_transaction("CASE-002-NORMAL", data.X_test[normal_idx])
Markdown(md_b)